# 4-Hour Google Colab Hands-on Tutorial  
## Supervised Learning Regression Project: House Price Prediction

**Project theme:** You are working as a junior ML engineer for a real-estate analytics company.  
Your task is to build a supervised machine learning regression model that predicts house prices using housing and location-related features.

This notebook is designed for a **4-hour instructor-led hands-on session**.

---

## What students will learn

By the end of this session, learners will be able to:

1. Explain supervised learning and regression in simple terms.
2. Understand the end-to-end machine learning lifecycle.
3. Load and explore a real regression dataset.
4. Prepare data for machine learning.
5. Train baseline and advanced regression models.
6. Evaluate regression models using MAE, RMSE, and R² score.
7. Tune a model using cross-validation.
8. Save the trained model and make predictions on new data.
9. Understand how to present ML project results.

# 4-Hour Tutorial Agenda

| Time | Module | Focus |
|---:|---|---|
| 0:00 - 0:25 | Module 1 | Introduction to supervised learning and regression |
| 0:25 - 0:55 | Module 2 | Problem statement and dataset understanding |
| 0:55 - 1:35 | Module 3 | Exploratory Data Analysis |
| 1:35 - 2:10 | Module 4 | Data preprocessing and train-test split |
| 2:10 - 2:55 | Module 5 | Model building: baseline, linear regression, tree models |
| 2:55 - 3:25 | Module 6 | Model evaluation and comparison |
| 3:25 - 3:50 | Module 7 | Hyperparameter tuning and model saving |
| 3:50 - 4:00 | Module 8 | Final discussion, assignment, and Q&A |

---

## Teaching approach

Use this structure for every module:

1. Explain the concept in layman terms.
2. Run the code cell.
3. Ask students to observe the output.
4. Discuss why the step is important in a real ML project.
5. Give a small task before moving to the next section.

# Module 1: Supervised Learning and Regression

## What is supervised learning?

Supervised learning means we train a machine learning model using examples where the correct answer is already known.

For example:

| Input Data | Correct Answer |
|---|---|
| House size, location, income level nearby | House price |
| Student study hours, attendance, previous marks | Exam score |
| Customer age, salary, past spending | Loan amount |

The model learns the relationship between input features and the target output.

---

## What is regression?

Regression is a type of supervised learning where the target value is a **number**.

Examples:

- Predicting house price
- Predicting sales revenue
- Predicting salary
- Predicting electricity consumption
- Predicting delivery time

In this tutorial, we will predict **house price**, so this is a **supervised regression problem**.

# Project Problem Statement

## Business Problem

A real-estate analytics company wants to estimate house prices based on district-level housing features such as income, house age, rooms, population, and location.

Currently, price estimation is manual and inconsistent. The business wants a machine learning model that can give a quick estimated house price.

---

## ML Problem

Build a supervised regression model to predict the median house value using historical housing data.

---

## Target variable

`MedHouseVal` = Median house value

In the California Housing dataset, this target value is represented in units of **$100,000**.

Example:

- `2.5` means approximately `$250,000`
- `4.0` means approximately `$400,000`

---

## Success Criteria

A good model should:

1. Have low prediction error.
2. Explain house value patterns reasonably.
3. Generalize well on unseen data.
4. Be easy to save and reuse for new predictions.

# End-to-End ML Project Lifecycle

In this hands-on tutorial, we will follow this flow:

1. Problem understanding  
2. Dataset loading  
3. Data understanding  
4. Exploratory Data Analysis  
5. Data preprocessing  
6. Train-test split  
7. Model training  
8. Model evaluation  
9. Model improvement  
10. Model saving  
11. New prediction  

This is the same high-level lifecycle followed in real-world machine learning projects.

# Module 2: Environment Setup

Run the following cell in Google Colab.

Most packages are already available in Colab, but this cell ensures required libraries are installed.

In [ ]:
# Install required libraries
!pip -q install pandas numpy matplotlib scikit-learn joblib

In [ ]:
# Import core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings

warnings.filterwarnings("ignore")

# Scikit-learn imports
from sklearn.datasets import fetch_california_housing, make_regression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully")

# Load the Dataset

We will use the California Housing dataset available through scikit-learn.

The dataset contains district-level housing data from California.  
Each row represents a block group/district, not one individual house.

If the dataset download fails in Colab due to internet issues, the notebook will create a synthetic fallback dataset so the tutorial can continue.

In [ ]:
# Load California Housing dataset with fallback option
try:
    housing = fetch_california_housing(as_frame=True)
    df = housing.frame.copy()
    print("California Housing dataset loaded successfully")
except Exception as e:
    print("Could not load California Housing dataset. Creating fallback synthetic regression dataset.")
    X, y = make_regression(
        n_samples=5000,
        n_features=8,
        noise=25,
        random_state=42
    )
    feature_names = [
        "MedInc", "HouseAge", "AveRooms", "AveBedrms",
        "Population", "AveOccup", "Latitude", "Longitude"
    ]
    df = pd.DataFrame(X, columns=feature_names)
    df["MedHouseVal"] = y / 100000
    df["MedHouseVal"] = df["MedHouseVal"] - df["MedHouseVal"].min() + 0.5

df.head()

In [ ]:
# Check dataset shape
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

## Dataset Column Meaning

| Column | Meaning |
|---|---|
| `MedInc` | Median income in the area |
| `HouseAge` | Median age of houses |
| `AveRooms` | Average number of rooms |
| `AveBedrms` | Average number of bedrooms |
| `Population` | Population of the area |
| `AveOccup` | Average number of occupants |
| `Latitude` | Latitude location |
| `Longitude` | Longitude location |
| `MedHouseVal` | Median house value, target variable |

---

## Student Task 1

Answer these questions:

1. Which column is the target variable?
2. Is this a classification or regression problem?
3. Why is house price prediction a regression problem?

# Module 3: Exploratory Data Analysis

Exploratory Data Analysis, or EDA, means understanding the data before building the model.

We will check:

1. Data types
2. Missing values
3. Summary statistics
4. Target distribution
5. Feature relationships
6. Correlation

In [ ]:
# Basic information about the dataset
df.info()

In [ ]:
# Check missing values
df.isnull().sum()

In [ ]:
# Summary statistics
df.describe().T

## Understanding the target variable

Before model building, always understand the target variable.

Here, the target is `MedHouseVal`.

In [ ]:
# Target variable summary
df["MedHouseVal"].describe()

In [ ]:
# Plot target distribution
plt.figure(figsize=(8, 5))
plt.hist(df["MedHouseVal"], bins=40)
plt.xlabel("Median House Value")
plt.ylabel("Frequency")
plt.title("Distribution of Target Variable: MedHouseVal")
plt.show()

## Feature Distribution

Now we will visualize the distribution of selected input features.

In [ ]:
# Plot important feature distributions
selected_features = ["MedInc", "HouseAge", "AveRooms", "Population"]

for col in selected_features:
    plt.figure(figsize=(8, 4))
    plt.hist(df[col], bins=40)
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {col}")
    plt.show()

## Relationship between income and house value

In real estate, income level of an area usually has a strong relationship with house value.

Let us check this visually.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["MedInc"], df["MedHouseVal"], alpha=0.3)
plt.xlabel("Median Income")
plt.ylabel("Median House Value")
plt.title("Median Income vs House Value")
plt.show()

## Correlation Analysis

Correlation helps us understand how strongly two numeric variables move together.

- Positive correlation: both increase together
- Negative correlation: one increases while the other decreases
- Near zero correlation: weak linear relationship

In [ ]:
# Correlation with target variable
correlation_with_target = df.corr(numeric_only=True)["MedHouseVal"].sort_values(ascending=False)
correlation_with_target

In [ ]:
# Correlation matrix using matplotlib
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Matrix")
plt.show()

## Student Task 2

Based on EDA, answer:

1. Which feature has the strongest positive correlation with house value?
2. Are there any columns with very large values or skewed distribution?
3. Why should we check missing values before model building?
4. Why should we not directly jump to model training without EDA?

# Module 4: Feature Engineering and Preprocessing

Most real ML projects need data preprocessing.

For this tutorial, we will add one simple categorical feature to demonstrate handling of categorical data.

We will create a `Region` column using latitude.

This is not required for the original dataset, but it helps students understand preprocessing pipelines with both numeric and categorical features.

In [ ]:
# Create a simple categorical feature from Latitude
def create_region(latitude):
    if latitude >= 38:
        return "North"
    elif latitude >= 35:
        return "Central"
    else:
        return "South"

df["Region"] = df["Latitude"].apply(create_region)

df[["Latitude", "Region"]].head()

In [ ]:
# Check region count
df["Region"].value_counts()

## Define Features and Target

Features are input columns.  
Target is the output column we want to predict.

In [ ]:
# Define target and features
target = "MedHouseVal"

X = df.drop(columns=[target])
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## Train-Test Split

We split data into:

- Training data: used to train the model
- Testing data: used to evaluate the model on unseen data

This helps us check whether the model can generalize.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## Build Preprocessing Pipeline

We have two types of columns:

1. Numeric columns: apply scaling
2. Categorical columns: apply one-hot encoding

Why scaling?

Some models, like Linear Regression and Ridge Regression, perform better when numeric features are scaled.

In [ ]:
# Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
# Preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created")

## Student Task 3

Answer these questions:

1. Why do we split data into training and testing sets?
2. What is the role of StandardScaler?
3. What is one-hot encoding?
4. Why do we use a pipeline instead of manually transforming data every time?

# Module 5: Model Building

We will train multiple regression models:

1. Dummy Regressor baseline
2. Linear Regression
3. Ridge Regression
4. Decision Tree Regressor
5. Random Forest Regressor

The goal is to compare models and select the best one.

## Evaluation Function

We will create one function to evaluate every model consistently.

Regression metrics:

| Metric | Meaning |
|---|---|
| MAE | Average absolute prediction error |
| RMSE | Penalizes large errors more strongly |
| R² Score | Explains how much variance the model captures |

For MAE and RMSE, lower is better.  
For R², higher is better.

In [ ]:
def evaluate_regression_model(model, X_test, y_test, model_name):
    # Evaluate a regression model using MAE, RMSE, and R2 score.
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results = {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    }

    return results

## 1. Baseline Model: Dummy Regressor

A baseline model gives a simple reference point.

Here, the dummy model always predicts the average house value.

Any useful ML model should perform better than this baseline.

In [ ]:
dummy_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="mean"))
])

dummy_model.fit(X_train, y_train)

dummy_results = evaluate_regression_model(
    dummy_model, X_test, y_test, "Dummy Regressor"
)

dummy_results

## 2. Linear Regression

Linear Regression tries to learn a straight-line relationship between input features and target value.

It is simple, fast, and highly interpretable.

In [ ]:
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

linear_results = evaluate_regression_model(
    linear_model, X_test, y_test, "Linear Regression"
)

linear_results

## 3. Ridge Regression

Ridge Regression is a regularized version of Linear Regression.

It helps reduce overfitting by controlling large coefficients.

In [ ]:
ridge_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

ridge_results = evaluate_regression_model(
    ridge_model, X_test, y_test, "Ridge Regression"
)

ridge_results

## 4. Decision Tree Regressor

Decision Tree learns rules from data.

Example:

- If income is high and location is good, predict higher price.
- If house age is high and rooms are low, predict lower price.

Decision trees are easy to understand but can overfit.

In [ ]:
tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42, max_depth=8))
])

tree_model.fit(X_train, y_train)

tree_results = evaluate_regression_model(
    tree_model, X_test, y_test, "Decision Tree"
)

tree_results

## 5. Random Forest Regressor

Random Forest builds many decision trees and combines their predictions.

It usually performs better than a single decision tree because it reduces overfitting.

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_results = evaluate_regression_model(
    rf_model, X_test, y_test, "Random Forest"
)

rf_results

# Module 6: Model Comparison

Now let us compare all models in one table.

In [ ]:
results_df = pd.DataFrame([
    dummy_results,
    linear_results,
    ridge_results,
    tree_results,
    rf_results
])

results_df.sort_values(by="RMSE")

In [ ]:
# Visual comparison of RMSE
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["RMSE"])
plt.xlabel("Model")
plt.ylabel("RMSE")
plt.title("Model Comparison by RMSE")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Visual comparison of R2 Score
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["R2 Score"])
plt.xlabel("Model")
plt.ylabel("R2 Score")
plt.title("Model Comparison by R2 Score")
plt.xticks(rotation=45)
plt.show()

## Error Analysis

A good ML engineer does not only check metrics.  
They also inspect predictions and errors.

Let us compare actual vs predicted values for the Random Forest model.

In [ ]:
# Actual vs predicted values
rf_predictions = rf_model.predict(X_test)

prediction_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": rf_predictions,
    "Error": y_test.values - rf_predictions
})

prediction_df.head(10)

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(7, 7))
plt.scatter(y_test, rf_predictions, alpha=0.3)
plt.xlabel("Actual House Value")
plt.ylabel("Predicted House Value")
plt.title("Actual vs Predicted House Values")

# Reference line
min_value = min(y_test.min(), rf_predictions.min())
max_value = max(y_test.max(), rf_predictions.max())
plt.plot([min_value, max_value], [min_value, max_value])

plt.show()

In [ ]:
# Residual/error distribution
plt.figure(figsize=(8, 5))
plt.hist(prediction_df["Error"], bins=40)
plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.show()

## Student Task 4

Answer these questions:

1. Which model has the lowest RMSE?
2. Which model has the highest R² score?
3. Why is the Dummy Regressor useful?
4. Why can Random Forest perform better than Linear Regression?
5. What does the Actual vs Predicted plot tell us?

# Cross-Validation

Train-test split gives one evaluation result.

Cross-validation evaluates the model on multiple splits of the training data.

This gives a more reliable idea of model performance.

In [ ]:
# Cross-validation for Random Forest
cv_scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)

rmse_scores = -cv_scores

print("Cross-validation RMSE scores:", rmse_scores)
print("Average CV RMSE:", rmse_scores.mean())
print("Standard deviation:", rmse_scores.std())

# Module 7: Hyperparameter Tuning

Hyperparameters are settings we choose before model training.

For Random Forest, examples are:

- Number of trees
- Maximum depth of trees
- Minimum samples required to split a node

We will use GridSearchCV to find a better combination.

In [ ]:
# Define a smaller grid to keep the tutorial fast in Colab
param_grid = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [8, 12, None],
    "model__min_samples_split": [2, 5]
}

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)

In [ ]:
# Evaluate tuned model
best_model = grid_search.best_estimator_

tuned_results = evaluate_regression_model(
    best_model,
    X_test,
    y_test,
    "Tuned Random Forest"
)

tuned_results

In [ ]:
# Final comparison including tuned model
final_results_df = pd.concat([
    results_df,
    pd.DataFrame([tuned_results])
], ignore_index=True)

final_results_df.sort_values(by="RMSE")

# Feature Importance

For tree-based models like Random Forest, we can check which features were most important for prediction.

This helps explain the model to business users.

In [ ]:
# Extract feature names after preprocessing
preprocessor_fitted = best_model.named_steps["preprocessor"]
model_fitted = best_model.named_steps["model"]

# Numeric feature names
num_features = numeric_features

# Categorical feature names after one-hot encoding
cat_encoder = preprocessor_fitted.named_transformers_["cat"].named_steps["onehot"]
cat_features = cat_encoder.get_feature_names_out(categorical_features).tolist()

all_feature_names = num_features + cat_features

# Feature importance
importance_df = pd.DataFrame({
    "Feature": all_feature_names,
    "Importance": model_fitted.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance_df.head(10)

In [ ]:
# Plot top 10 feature importances
top_features = importance_df.head(10)

plt.figure(figsize=(9, 5))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 10 Feature Importances")
plt.gca().invert_yaxis()
plt.show()

# Save the Final Model

In real projects, after training a model, we save it.

The saved model can later be loaded inside:

- A Flask API
- A FastAPI service
- A Streamlit app
- A batch prediction job
- A cloud deployment pipeline

In [ ]:
# Save final model
model_filename = "house_price_regression_model.pkl"
joblib.dump(best_model, model_filename)

print(f"Model saved as {model_filename}")

## Download the Model from Colab

Run this cell if you want to download the model file to your local computer.

In [ ]:
from google.colab import files

files.download(model_filename)

# Load the Saved Model and Make a New Prediction

Now we will simulate a real-world scenario.

A user gives housing details, and the model predicts the approximate house value.

In [ ]:
# Load saved model
loaded_model = joblib.load(model_filename)

# Create one sample input row
sample_house = pd.DataFrame([{
    "MedInc": 5.0,
    "HouseAge": 20.0,
    "AveRooms": 6.0,
    "AveBedrms": 1.1,
    "Population": 1200.0,
    "AveOccup": 3.0,
    "Latitude": 34.2,
    "Longitude": -118.4,
    "Region": "South"
}])

sample_house

In [ ]:
# Predict house value
predicted_value = loaded_model.predict(sample_house)[0]

print("Predicted median house value:", round(predicted_value, 2))
print("Approximate value in dollars:", "$" + str(round(predicted_value * 100000, 2)))

# Module 8: Final Project Summary

## What we completed

In this 4-hour hands-on project, we completed an end-to-end supervised learning regression workflow:

1. Understood the business problem
2. Defined the ML problem
3. Loaded the dataset
4. Performed EDA
5. Created features
6. Split data into train and test sets
7. Built preprocessing pipelines
8. Trained multiple regression models
9. Compared model performance
10. Performed cross-validation
11. Tuned hyperparameters
12. Saved the final model
13. Loaded the model and predicted on new data

---

## Final Business Explanation

The final model can estimate house prices based on district-level housing and location data.

This can help a real-estate analytics team:

- Quickly estimate property value
- Compare areas
- Identify important price-driving factors
- Build a future price recommendation system

# Final Student Assignment

## Assignment 1: Improve the Model

Try the following:

1. Change Random Forest hyperparameters.
2. Add a new feature using existing columns.
3. Try GradientBoostingRegressor.
4. Compare all models again.

---

## Assignment 2: Build a Mini Prediction App

Create a simple Streamlit app where the user enters:

- Median income
- House age
- Average rooms
- Average bedrooms
- Population
- Average occupancy
- Latitude
- Longitude
- Region

The app should output predicted house value.

---

## Assignment 3: Business Presentation

Create a 5-slide presentation:

1. Problem statement
2. Dataset overview
3. EDA insights
4. Model comparison
5. Final recommendation

# Interview and Viva Questions

1. What is supervised learning?
2. What is regression?
3. What is the difference between regression and classification?
4. What is the target variable in this project?
5. Why do we split data into training and testing sets?
6. What is MAE?
7. What is RMSE?
8. What is R² score?
9. Why do we need preprocessing?
10. What is overfitting?
11. Why is cross-validation useful?
12. What is hyperparameter tuning?
13. Why do we save a trained model?
14. Which model performed best in this project and why?
15. How would you deploy this model in real life?

# Optional Extension: Try Gradient Boosting

This is an optional extra model for students who finish early.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

gb_model.fit(X_train, y_train)

gb_results = evaluate_regression_model(
    gb_model,
    X_test,
    y_test,
    "Gradient Boosting"
)

gb_results

# Trainer Notes

## Suggested explanation style

Use the following simple analogy:

> Machine learning is like teaching a student using examples.  
> In supervised learning, every example has a question and an answer.  
> In regression, the answer is a number.

Example:

- Question: What will be the house price?
- Inputs: income, house age, rooms, location
- Answer: predicted house value

---

## Common beginner mistakes to highlight

1. Training and testing on the same data
2. Ignoring missing values
3. Looking only at accuracy-like metrics
4. Not comparing with a baseline
5. Not checking overfitting
6. Not saving the final model
7. Not explaining results in business language

# References for Further Learning

1. Scikit-learn documentation: Regression models, pipelines, preprocessing, and model evaluation
2. Google Colab documentation: Running Python notebooks in the browser
3. Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow by Aurélien Géron
4. Kaggle Learn: Intro to Machine Learning
5. Machine Learning Mastery: Regression metrics and model evaluation tutorials